# Error Analysis
Using the best model we have achieved so far, we can see the mistakes it is making on the final test set

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
from collections import Counter
from datasets import load_dataset
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## Data Preparation and Label Consolidation

In [2]:
print("\n" + "="*80)
print("1. DATA LOADING & PREPARATION")
print("="*80)

# MAPEO OFICIAL
EVASION_LABELS = {
    0: 'Claims ignorance', 1: 'Clarification', 2: 'Declining to answer',
    3: 'Deflection', 4: 'Dodging', 5: 'Explicit',
    6: 'General', 7: 'Implicit', 8: 'Partial/half-answer'
}
LABEL2ID = {v: k for k, v in EVASION_LABELS.items()}
label_names = [EVASION_LABELS[i] for i in sorted(EVASION_LABELS.keys())]

# Cargar dataset
ds = load_dataset("ailsntua/QEvasion")
train_df = ds['train'].to_pandas()
test_set_raw = ds['test']

# --- CORRECCIÓN 1: BLINDAR Y_TRAIN ---
# Verificar si train_df['evasion_label'] es texto y convertirlo si es necesario
if train_df['evasion_label'].dtype == 'O' or isinstance(train_df['evasion_label'].iloc[0], str):
    print("Detected string labels in Train set. Converting to IDs...")
    train_df['evasion_label'] = train_df['evasion_label'].map(LABEL2ID)

train_df['evasion_label'] = train_df['evasion_label'].astype(int)
print("Train labels type checked: IDs (int)")

# Función de Voto por Mayoría
def consolidate_labels(example):
    votes = [example['annotator1'], example['annotator2'], example['annotator3']]
    counts = Counter(votes)
    most_common = counts.most_common()
    if most_common[0][1] > 1:
        final_label_txt = most_common[0][0]
    else:
        final_label_txt = random.choice(votes)
    return {'evasion_label_txt': final_label_txt}

# Procesar Test Set
print("Consolidating Test Set Labels...")
test_set_processed = test_set_raw.map(consolidate_labels)
test_df = test_set_processed.to_pandas()

# --- CORRECCIÓN 2: BLINDAR Y_TEST ---
print("Mapping Test Labels to Numeric IDs...")
# Asegurarnos de limpiar espacios en blanco por si acaso
test_df['evasion_label_txt'] = test_df['evasion_label_txt'].str.strip()
test_df['evasion_label'] = test_df['evasion_label_txt'].map(LABEL2ID)

# Eliminar errores de mapeo
if test_df['evasion_label'].isnull().any():
    n_dropped = test_df['evasion_label'].isnull().sum()
    print(f"Warning: Dropping {n_dropped} rows with unmapped labels.")
    test_df = test_df.dropna(subset=['evasion_label'])

test_df['evasion_label'] = test_df['evasion_label'].astype(int)

# Context Features
for df in [train_df, test_df]:
    df['sub_q_context'] = df['question'].fillna('') + " [SEP] " + df['interview_answer'].fillna('')

print(f"Final Train size: {len(train_df)}")
print(f"Final Test size:  {len(test_df)}")


1. DATA LOADING & PREPARATION


Detected string labels in Train set. Converting to IDs...
Train labels type checked: IDs (int)
Consolidating Test Set Labels...


Map:   0%|          | 0/308 [00:00<?, ? examples/s]

Mapping Test Labels to Numeric IDs...
Final Train size: 3448
Final Test size:  308


## Model Training

In [ ]:
print("\n" + "="*80)
print("2. TRAINING BEST MODEL")
print("="*80)

# Hiperparámetros Fase 3
BEST_C = 4.57
BEST_SOLVER = 'lbfgs'
NGRAM_RANGE = (1, 2)

X_train_text = train_df['sub_q_context']
y_train = train_df['evasion_label'] # INT

X_test_text = test_df['sub_q_context']
y_test = test_df['evasion_label']   # INT

print("Vectorizing...")
vectorizer = CountVectorizer(ngram_range=NGRAM_RANGE, min_df=2, max_df=0.95, stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train_text)
X_test_vec = vectorizer.transform(X_test_text)

print("Training Logistic Regression...")
clf = LogisticRegression(C=BEST_C, solver=BEST_SOLVER, max_iter=1000, multi_class='multinomial', random_state=SEED)
clf.fit(X_train_vec, y_train)

print("Predicting...")
y_pred = clf.predict(X_test_vec) # Predecirá INTs


2. TRAINING BEST MODEL
Vectorizing...
Training Logistic Regression...


/home/alumno/py313ml/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


## Confusion Matrix

In [ ]:
print("\n" + "="*80)
print("3. CONFUSION MATRIX")
print("="*80)

# Ahora ambos son ints, no debería fallar
print(classification_report(y_test, y_pred, target_names=label_names))

cm = confusion_matrix(y_test, y_pred, labels=sorted(EVASION_LABELS.keys()), normalize='true')
plt.figure(figsize=(12, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot(cmap='Blues', values_format='.2f', ax=plt.gca())
plt.title("Normalized Confusion Matrix (Test Set)", fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.show()

## Error Analysis 
### Discriminative Features



In [ ]:
print("\n" + "="*80)
print("4. DISCRIMINATIVE FEATURES (Top N-grams)")
print("="*80)

feature_names = vectorizer.get_feature_names_out()

def plot_top_features(class_idx, class_name, top_n=10):
    coefs = clf.coef_[class_idx]
    top_indices = np.argsort(coefs)[-top_n:]
    top_features = [feature_names[i] for i in top_indices]
    top_weights = coefs[top_indices]
    
    plt.figure(figsize=(8, 4))
    plt.barh(range(top_n), top_weights, color='steelblue')
    plt.yticks(range(top_n), top_features)
    plt.xlabel('Coefficient Weight')
    plt.title(f"Top Predictive Features for: '{class_name}'")
    plt.tight_layout()
    plt.show()
    print(f"Top features for {class_name}: {top_features[::-1]}")

# Analizamos 3 clases interesantes (puedes cambiar los índices)
# 0: Ignorance, 4: Dodging, 5: Explicit
for idx in [0, 4, 5]: 
    plot_top_features(idx, EVASION_LABELS[idx])

### Qualitative Failure Examples

In [ ]:
print("\n" + "="*80)
print("5. QUALITATIVE FAILURE ANALYSIS (Manual Inspection)")
print("="*80)

# Preparar DataFrame de Análisis
analysis_df = test_df.copy()
analysis_df['true_label_id'] = y_test.values
analysis_df['pred_label_id'] = y_pred
analysis_df['true_label_txt'] = analysis_df['true_label_id'].map(EVASION_LABELS)
analysis_df['pred_label_txt'] = analysis_df['pred_label_id'].map(EVASION_LABELS)
analysis_df['is_error'] = analysis_df['true_label_id'] != analysis_df['pred_label_id']

errors_df = analysis_df[analysis_df['is_error']]
print(f"Total Errors: {len(errors_df)} / {len(test_df)}")

def print_example(row):
    print(f"\n[Error Example]")
    print(f"Q: {row['question']}")
    print(f"A: {row['interview_answer']}")
    print(f"TRUE: {row['true_label_txt']} | PRED: {row['pred_label_txt']}")
    print("-" * 50)

# 1. CLARIFICATION -> DODGING (Error masivo: 75% en matriz)
print("\n>>> TYPE A: 'Clarification' misclassified as 'Dodging'")
type_a = errors_df[(errors_df['true_label_txt'] == 'Clarification') & 
                   (errors_df['pred_label_txt'] == 'Dodging')]
if not type_a.empty:
    print_example(type_a.iloc[0])
else:
    print("No examples found for this specific error.")

# 2. IMPLICIT -> EXPLICIT (Error frecuente: 44% en matriz)
print("\n>>> TYPE B: 'Implicit' misclassified as 'Explicit'")
type_b = errors_df[(errors_df['true_label_txt'] == 'Implicit') & 
                   (errors_df['pred_label_txt'] == 'Explicit')]
if not type_b.empty:
    print_example(type_b.iloc[0])
else:
    print("No examples found for this specific error.")

# 3. EXPLICIT -> DODGING (Error común por longitud o retórica)
print("\n>>> TYPE C: 'Explicit' misclassified as 'Dodging'")
type_c = errors_df[(errors_df['true_label_txt'] == 'Explicit') & 
                   (errors_df['pred_label_txt'] == 'Dodging')]
if not type_c.empty:
    print_example(type_c.iloc[0])

# 4. PARTIAL/HALF-ANSWER -> EXPLICIT (Error de sutileza)
print("\n>>> TYPE D: 'Partial/half-answer' misclassified as 'Explicit'")
type_d = errors_df[(errors_df['true_label_txt'] == 'Partial/half-answer') & 
                   (errors_df['pred_label_txt'] == 'Explicit')]
if not type_d.empty:
    print_example(type_d.iloc[0])

# 5. RANDOM SAMPLE (Para variedad)
print("\n>>> TYPE E: Random Error Sample")
if not errors_df.empty:
    random_error = errors_df.sample(1, random_state=SEED).iloc[0]
    print_example(random_error)

print("\nAnalysis Complete.")